# Lab 20: XAI and Deployment

**Module 20** | Read `notes/20-shap-lime-xai-deployment.pdf` before running this notebook.

Train a fraud classifier, explain predictions with SHAP and LIME, and expose a Flask /predict endpoint.

Run every cell top to bottom. Code is complete, no fill-in sections. Markdown cells explain what each block does and why.


## Runtime device

PyTorch can run on your NVIDIA GPU through CUDA or fall back to CPU. GPU execution moves matrix operations off the host and typically speeds up neural network training by a large factor. The next cell detects what is available and prints the active device so you know whether to expect fast training or should keep batch sizes small.


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"CUDA enabled, {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("CUDA not available, using CPU. Labs still run; training may take longer.")


# Lab 20: XAI and Deployment

**Module 20** | Read `notes/20-shap-lime-xai-deployment.pdf` before running this notebook.

Train a fraud classifier, explain predictions with SHAP and LIME, and expose a Flask /predict endpoint.

Run every cell top to bottom. Code is complete, no fill-in sections. Markdown cells explain what each block does and why.


## Explainable AI and Model Deployment

Machine learning models used for fraud detection must be **interpretable** and **deployable**.
This lab covers three layers:

1. **Train** a scikit-learn Random Forest on the credit-fraud sample (V1 to V28 features).
2. **Explain** predictions with **SHAP** (global/local tree attributions) and **LIME** (local linear approximation).
3. **Serve** the model through a minimal **Flask** REST API at `POST /predict`.

The Flask app is defined in-notebook and tested with Flask's **test client** so the notebook does not block waiting for a server process.
For production you would run `flask run` or gunicorn in a separate terminal/thread.


In [ ]:
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "scripts"))
from runtime_env import configure_runtime

configure_runtime(quiet=True)

parquet_path = ROOT / "datasets" / "credit_fraud_sample.parquet"
csv_path = ROOT / "datasets" / "credit_fraud_sample.csv"
if parquet_path.exists():
    df = pd.read_parquet(parquet_path)
else:
    df = pd.read_csv(csv_path)

feature_cols = [f"V{i}" for i in range(1, 29)]
X = df[feature_cols].values
y = df["Class"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)
print(classification_report(y_test, clf.predict(X_test), target_names=["legit", "fraud"]))

model_path = ROOT / "models" / "credit_fraud_rf.joblib"
model_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump({"model": clf, "feature_cols": feature_cols}, model_path)
print(f"Saved model bundle to {model_path}")


## SHAP, TreeExplainer

SHAP (SHapley Additive exPlanations) assigns each feature an contribution to the prediction for tree-based models.
`TreeExplainer` is exact and fast for Random Forests.

We explain **10 test rows** and inspect the shape of the resulting arrays.
For binary classification, SHAP returns a list with one array per class; class **1** corresponds to the fraud label.


In [ ]:
import shap

explainer = shap.TreeExplainer(clf)
n_explain = min(10, len(X_test))
sample_idx = np.arange(n_explain)
shap_values = explainer.shap_values(X_test[sample_idx])

print(f"SHAP values computed for {n_explain} test rows")
if isinstance(shap_values, list):
    fraud_shap = np.array(shap_values[1])
    print(f"SHAP array shape (fraud class): {fraud_shap.shape}")
    print(f"Mean |SHAP| per feature (fraud class, first row): "
          f"{np.abs(fraud_shap[0]).mean():.4f}")
else:
    print(f"SHAP array shape: {np.array(shap_values).shape}")


## LIME, local interpretable explanation

LIME fits a sparse linear model in the neighborhood of a single prediction to show which features pushed toward fraud vs legitimate.
We explain **one test row** and print the top weighted features.


In [ ]:
import lime.lime_tabular

lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    X_train,
    feature_names=feature_cols,
    class_names=["legit", "fraud"],
    mode="classification",
    random_state=42,
)

lime_row = 0
lime_exp = lime_explainer.explain_instance(
    X_test[lime_row],
    clf.predict_proba,
    num_features=8,
)

print(f"LIME explanation for test row {lime_row} (true label: {y_test[lime_row]})")
print(f"Predicted prob(fraud): {clf.predict_proba(X_test[lime_row : lime_row + 1])[0][1]:.4f}")
print("Top features:")
for feat, weight in lime_exp.as_list():
    print(f"  {feat}: {weight:+.4f}")


## Flask inference API

The app exposes `POST /predict` expecting JSON:

```json
{"instances": [[v1, v2, ..., v28], ...]}
```

We test with Flask's built-in **test client**; no `app.run()` call, so the notebook finishes immediately.
To serve live traffic, run the app in a **separate terminal** (or background thread):

```python
# Optional: run outside the notebook:
# app.run(host="127.0.0.1", port=8080, debug=False)
```


In [ ]:
from flask import Flask, jsonify, request

app = Flask(__name__)
_bundle = joblib.load(model_path)
_model = _bundle["model"]
_feature_cols = _bundle["feature_cols"]


@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok", "features": len(_feature_cols)})


@app.route("/predict", methods=["POST"])
def predict():
    payload = request.get_json(force=True, silent=True) or {}
    rows = payload.get("instances")
    if not rows:
        return jsonify({"error": "Provide JSON: {'instances': [[V1,..,V28], ...]}"}), 400

    X_in = np.asarray(rows, dtype=float)
    if X_in.ndim == 1:
        X_in = X_in.reshape(1, -1)
    if X_in.shape[1] != len(_feature_cols):
        return jsonify({
            "error": f"Expected {len(_feature_cols)} features per row, got {X_in.shape[1]}"
        }), 400

    probs = _model.predict_proba(X_in)
    preds = _model.predict(X_in)
    return jsonify({
        "predictions": preds.astype(int).tolist(),
        "probabilities": probs.tolist(),
        "feature_cols": _feature_cols,
    })


with app.test_client() as client:
    health_resp = client.get("/health")
    print("GET /health:", health_resp.status_code, health_resp.get_json())

    test_row = X_test[0].tolist()
    pred_resp = client.post("/predict", json={"instances": [test_row]})
    print("POST /predict status:", pred_resp.status_code)
    print("POST /predict body:", pred_resp.get_json())

    bad_resp = client.post("/predict", json={"instances": [[1.0, 2.0]]})
    print("POST /predict (bad shape) status:", bad_resp.status_code)
    print("POST /predict (bad shape) body:", bad_resp.get_json())


## Summary

| Component | Purpose |
|-----------|---------|
| Random Forest | Strong baseline on tabular fraud data |
| SHAP TreeExplainer | Exact Shapley values for tree models (batch of 10 rows) |
| LIME | Human-readable local explanation for a single transaction |
| Flask `/predict` | Production-style HTTP inference (tested in-process here) |

**Next steps:** containerize the Flask app (Docker), add input validation schema (pydantic), and log prediction latency plus explanation metadata to MLflow for audit trails.
